In [ ]:
import streamlit as st
import pandas as pd
import os
import configparser
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F

# Configuración de la conexión a Snowflake.
config_path = os.path.join(os.environ['USERPROFILE'], '.snowsql', 'config')
config = configparser.ConfigParser()
config.read(config_path)

# Se obtienen los parámetros de conexión desde el archivo de configuración.
try:
    account = config['connections.example']['accountname']
    user = config['connections.example']['username']
    password = config['connections.example']['password']
except KeyError as e:
    print(f'Error: {e}')

# Se definen los parámetros de conexión a Snowflake.
connection_parameters = {
    "account": account,
    "user": user,
    "password": password,
    "warehouse": "COMPUTE_WH",
    "database": "DEVMOON_SAMPLE",
    "schema": "PUBLIC",
}

# Se crea la sesión de Snowpark utilizando los parámetros de conexión.
try:
    session = Session.builder.configs(connection_parameters).create()
    print('Conexión exitosa con Snowpark.')
except Exception as e:
    print(f'Error: {e}')

Conexión exitosa con Snowpark.


In [ ]:
# Se seleccionan las tablas de Snowflake que se utilizarán en el análisis.

orders_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS')
customer_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER')
nation_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION')
region_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION')


In [ ]:
# Se filtran los datos para obtener las órdenes de clientes de Estados Unidos.

usa_region_key = region_table.filter(
    F.col('R_NAME') == 'AMERICA').select('R_REGIONKEY')
usa_nation_key = nation_table.filter(
    (F.col('N_NAME') == 'UNITED STATES') & (F.col('N_REGIONKEY').isin(usa_region_key))).select('N_NATIONKEY')
usa_customer_key = customer_table.filter(
    F.col('C_NATIONKEY').isin(usa_nation_key)).select('C_CUSTKEY')
usa_orders = orders_table.filter(F.col('O_CUSTKEY').isin(usa_customer_key))

In [ ]:
# Se agrupan los datos por fecha de orden y se calcula la suma de las ventas totales para cada día.

daily_salea_sndf = usa_orders.group_by(
    F.to_date(F.col('O_ORDERDATE')).alias('ORDER_DATE')
).agg(F.sum('O_TOTALPRICE').alias('TOTAL_SALES'))

daily_salea_sndf.count()

2406

In [ ]:
# Se convierte el DataFrame de Snowpark a un DataFrame de Pandas y se ordena por fecha de orden.

daily_salea_pdf = daily_salea_sndf.to_pandas()
daily_salea_pdf = daily_salea_pdf.sort_values(by = 'ORDER_DATE')

In [6]:
import plotly.express as px

In [ ]:
# Se crea un gráfico de línea utilizando Plotly Express para visualizar las ventas diarias totales en Estados Unidos.

px.line(daily_salea_pdf, x = 'ORDER_DATE', y = 'TOTAL_SALES',
        title='Ventas Diarias totales en Estados Unidos.')

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
# Se prepara el conjunto de datos para el modelo de regresión lineal.

daily_salea_pdf['ORDER_DATE_NUM'] = pd.to_datetime(daily_salea_pdf['ORDER_DATE']).astype('int64') // 10**9 # Convierte las fechas a númerico (timestamp segundos)
X = daily_salea_pdf[['ORDER_DATE_NUM']] # Caracteristica: Fecha númerica
y = daily_salea_pdf['TOTAL_SALES'] # Variable objetivo: Ventas totales

In [ ]:
# Se divide el conjunto de datos en conjuntos de entrenamiento y prueba.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42 )

In [ ]:
# Se crea y entrena el modelo de regresión lineal.

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[0.]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['ORDER_DATE_NUM']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,3.433e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


In [ ]:
# Se realizan predicciones utilizando el conjunto de prueba y se calcula el error cuadrático medio (MSE) para evaluar el rendimiento del modelo.

y_pred = model.predict(X_test)

In [ ]:
# Se calcula el error cuadrático medio (MSE) para evaluar el rendimiento del modelo.

mse = mean_squared_error(y_test, y_pred)
print(f'Error Cuadrático Medio en el conjunto de prueba: {mse}')

Error Cuadrático Medio en el conjunto de prueba: 739068361332.9396


In [ ]:
# Se crea un DataFrame para comparar las ventas reales con las ventas predichas por el modelo.

predictions_df = pd.DataFrame({'Fecha': daily_salea_pdf['ORDER_DATE'].iloc[X_test.index].values,
                               'Ventas Reales': y_test.values,
                               'Ventas Predichas': y_pred})

In [ ]:
# Se agrupan los datos por fecha y se calcula la suma de las ventas reales y predichas para cada día.

predictions_df = predictions_df.groupby('Fecha').agg({
    'Ventas Reales':'sum',
    'Ventas Predichas':'sum'
}).reset_index()

In [ ]:
# Se crea un gráfico de línea utilizando Plotly Express para visualizar las ventas reales y predichas en el conjunto de prueba.

px.line(predictions_df, x='Fecha', y=['Ventas Reales', 'Ventas Predichas'],
        title='Predicciones de Ventas vs. Ventas Reales (Conjunto de Prueba)', 
        render_mode='svg')

In [17]:
import joblib
import os

In [ ]:
# Se crea un directorio para almacenar el modelo entrenado en Snowflake, si no existe.

session.sql('CREATE OR REPLACE STAGE my_ml_models_stage').collect()

[Row(status='Stage area MY_ML_MODELS_STAGE successfully created.')]

In [ ]:
# Se guarda el modelo entrenado localmente y se sube al stage de Snowflake.

model_stage_path = '@my_ml_models_stage/linear_regression_sales_model.joblib' #Reemplaza 'my_ml_model_stage'
joblib.dump(model, 'model.joblib') # Guarda localmente en la sesion de Snowdight (memoria)
session.file.put("model.joblib", model_stage_path, overwrite=True) # Sube al stage

[PutResult(source='model.joblib', target='model.joblib.gz', source_size=895, target_size=592, source_compression='NONE', target_compression='GZIP', status='UPLOADED', message='')]

In [ ]:
# Se descarga el modelo desde el stage de Snowflake y se carga en memoria para su uso posterior.

download_files = session.file.get(model_stage_path, './')
loaded_model = joblib.load(download_files[0].file)

In [ ]:
# Se generan predicciones para los próximos 90 días utilizando el modelo cargado.

last_date_numeric = X['ORDER_DATE_NUM'].max()
future_dates_numeric = [last_date_numeric + (i * 86400) for i in range(1,91)]
future_dates_dt = pd.to_datetime(future_dates_numeric, unit='s')
future_dates_df = pd.DataFrame({'ORDER_DATE_NUM': future_dates_numeric, 'ORDER_DATE': future_dates_dt})

future_predictions = loaded_model.predict(future_dates_df[['ORDER_DATE_NUM']])

In [ ]:
# Se crea un DataFrame para almacenar las fechas futuras y las ventas predichas correspondientes.

future_predictions_df = pd.DataFrame({'Fecha Futura': future_dates_df['ORDER_DATE'],
                                'Ventas Predichas': future_predictions
                                })

In [ ]:
# Se crea un gráfico de línea utilizando Plotly Express para visualizar las predicciones de ventas futuras.

fig_future_predictions = px.line(future_predictions_df, x = 'Fecha Futura', y = 'Ventas Predichas',
                                 title='Predicciones de Ventas Futuras (3 meses)', render_mode='svg')

fig_future_predictions